In [1]:
import pandas as pd
df = pd.read_pickle('coef_clean_two.pkl')
df.head()

,insurance_type,label,prev_readmit_group,los_group,risk_score_bin,dc_location,primary_dx_tier,age_bin
0,Medicare/Medicaid,1,1,6 to 8,7,HH/SNF,lower,2.0
1,Private,1,2,6 to 8,7,HH/SNF,lower,2.0
2,Medicare/Medicaid,1,2,6 to 8,7,HH/SNF,lower,3.0
3,Medicare/Medicaid,1,2,9+,7,HH/SNF,higher,3.0
4,Uninsured,1,1,9+,6,HH/SNF,higher,2.0


In [2]:
feature_cols = [
    'insurance_type',
    'prev_readmit_group',
    'los_group',
    'risk_score_bin',
    'dc_location',
    'primary_dx_tier',
    'age_bin'
]

In [3]:
df[feature_cols] = df[feature_cols].astype('category')

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

X = df[feature_cols]
y = df['label']

X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# logistic regression
model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)

# predicted probabilities
lr_probs = model.predict_proba(X_test)[:, 1]

# confusion matrix at 0.50 threshold
lr_preds = (lr_probs >= 0.5).astype(int)

print("Logistic Regression Confusion Matrix:")
print(confusion_matrix(y_test, lr_preds))

print("\nLogistic Regression Classification Report:")
print(classification_report(y_test, lr_preds))


Logistic Regression Confusion Matrix:
[[ 165  186]
 [  97 1152]]

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.47      0.54       351
           1       0.86      0.92      0.89      1249

    accuracy                           0.82      1600
   macro avg       0.75      0.70      0.71      1600
weighted avg       0.81      0.82      0.81      1600



In [5]:
coef = pd.Series(model.coef_[0], index=X_encoded.columns)

# Sort alphabetically by feature name
coef_sorted_by_name = coef.sort_index()

coef_sorted_by_name

age_bin_2.0                 0.463904
age_bin_3.0                 0.915967
dc_location_Home/Rehab     -0.214219
insurance_type_Private     -0.072597
insurance_type_Uninsured    0.059116
los_group_6 to 8           -0.064585
los_group_9+                0.186029
prev_readmit_group_1        0.384861
prev_readmit_group_2        0.732416
primary_dx_tier_lower      -0.275650
risk_score_bin_2            0.347996
risk_score_bin_3            0.719197
risk_score_bin_4            0.972702
risk_score_bin_5            1.182070
risk_score_bin_6            1.678588
risk_score_bin_7            2.661592
dtype: float64

In [6]:
def show_raw_coefs_for(model, feature):
    coefs = pd.Series(model.coef_.flatten(), index=model.feature_names_in_)
    # return coefs[coefs.index.str.startswith(feature + "_")]
    print('baseline = 0.0000')
    print(coefs[coefs.index.str.startswith(feature + "_")])

In [7]:
group = 'los_group'
bins = df[group].value_counts()
print(bins)
show_raw_coefs_for(model, group)

los_group
6 to 8       4205
9+           2851
5 or less     944
Name: count, dtype: int64
baseline = 0.0000
los_group_6 to 8   -0.064585
los_group_9+        0.186029
dtype: float64


In [8]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

# 1. Initialize the model
gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3
)

# 2. Fit the model
gb.fit(X_train, y_train)

# 3. Predict probabilities
gb_probs = gb.predict_proba(X_test)[:, 1]

# 4. Score (AUC)
gb_auc = roc_auc_score(y_test, gb_probs)
print("Gradient Boosting AUC:", gb_auc)

# 5. Confusion matrix at 0.5 threshold
gb_preds = (gb_probs >= 0.5).astype(int)
print("Confusion Matrix:\n", confusion_matrix(y_test, gb_preds))

# 6. Optional: precision/recall/F1
print(classification_report(y_test, gb_preds))


Gradient Boosting AUC: 0.8418312997976729
Confusion Matrix:
 [[ 184  167]
 [ 118 1131]]
              precision    recall  f1-score   support

           0       0.61      0.52      0.56       351
           1       0.87      0.91      0.89      1249

    accuracy                           0.82      1600
   macro avg       0.74      0.71      0.73      1600
weighted avg       0.81      0.82      0.82      1600



In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

# 1. Initialize the model
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

# 2. Fit the model
rf.fit(X_train, y_train)

# 3. Predict probabilities
rf_probs = rf.predict_proba(X_test)[:, 1]

# 4. Score (AUC)
rf_auc = roc_auc_score(y_test, rf_probs)
print("Random Forest AUC:", rf_auc)

# 5. Confusion matrix at 0.5 threshold
rf_preds = (rf_probs >= 0.5).astype(int)
print("Confusion Matrix:\n", confusion_matrix(y_test, rf_preds))

# 6. Optional: precision/recall/F1
print(classification_report(y_test, rf_preds))


Random Forest AUC: 0.8428395137762632
Confusion Matrix:
 [[ 164  187]
 [ 111 1138]]
              precision    recall  f1-score   support

           0       0.60      0.47      0.52       351
           1       0.86      0.91      0.88      1249

    accuracy                           0.81      1600
   macro avg       0.73      0.69      0.70      1600
weighted avg       0.80      0.81      0.81      1600



In [10]:
from sklearn.metrics import confusion_matrix

thresholds = [0.4, 0.5, 0.6]

for t in thresholds:
    preds = (lr_probs >= t).astype(int)
    cm = confusion_matrix(y_test, preds)
    print(f"\nThreshold = {t}")
    print(cm)

# [TN, FP]
# [FN, TP]



Threshold = 0.4
[[ 128  223]
 [  62 1187]]

Threshold = 0.5
[[ 165  186]
 [  97 1152]]

Threshold = 0.6
[[ 215  136]
 [ 153 1096]]


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X = df[feature_cols]
y = df['label']

X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)
model.score(X_test, y_test)

0.823125

In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_at_threshold(threshold):
    preds = (lr_probs >= threshold).astype(int)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    return acc, prec, rec, f1

for t in [0.4, 0.5, 0.6]:
    acc, prec, rec, f1 = evaluate_at_threshold(t)
    print(f"\nThreshold = {t}")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")



Threshold = 0.4
Accuracy:  0.8219
Precision: 0.8418
Recall:    0.9504
F1 Score:  0.8928

Threshold = 0.5
Accuracy:  0.8231
Precision: 0.8610
Recall:    0.9223
F1 Score:  0.8906

Threshold = 0.6
Accuracy:  0.8194
Precision: 0.8896
Recall:    0.8775
F1 Score:  0.8835
